In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *

In [0]:
#Declaring Source and Destination Paths
rawpath_promotion = '/mnt/raw/MasterData/Promotion/Promotion.csv'
rawpath_invoice = '/mnt/raw/MasterData/PatientClassification/PatientClassification.csv'

silverpath_promotion = '/mnt/silver/DIMPromotion'
silverpath_patientclassification = '/mnt/silver/DIMPatientClassification'

# Promotion

In [0]:
promotion_df = (spark.read.options(header='true', delimiter=',').csv(rawpath_promotion)\
                           .withColumn("EffectiveDate", to_date('EffectiveDate', "M/d/yyyy").alias("date"))\
                           .withColumn("TerminationDate", to_date('TerminationDate', "M/d/yyyy").alias("date")))\
                           .withColumn('PromotionUUID',expr('uuid()'))\
                           .withColumn('CreatedBy',lit('ADB_Promotion'))\
                           .withColumn('CreatedDate',current_timestamp())\
                           .withColumn('UpdatedBy',lit('ADB_Promotion'))\
                           .withColumn('UpdatedDate',current_timestamp())\
                           .select('PromotionUUID','PromotionName','PromotionComments','EffectiveDate','TerminationDate','DeviceTypeCd',
                                    'CreatedDate','CreatedBy','UpdatedDate','UpdatedBy','Active')


promotion_df.display()

In [0]:
targetDF = DeltaTable.forPath(spark, silverpath_promotion)  
(targetDF.alias("tgt")
  .merge(promotion_df.alias("src"), "src.PromotionName = tgt.PromotionName and tgt.Active = 1")
  .whenMatchedUpdate(
    set = {
        "tgt.PromotionComments": "src.PromotionComments",
        "tgt.TerminationDate": "src.TerminationDate",
        "tgt.DeviceTypeCd": "src.DeviceTypeCd",
        "tgt.Active": "src.Active",
        "tgt.UpdatedDate": "src.UpdatedDate",
        "tgt.UpdatedBy": "src.UpdatedBy"
           }
  )
  .whenNotMatchedInsertAll()
  .execute()
)

# PatientClassification

In [0]:
PromotionUUID= spark.sql("select PromotionUUID from promotion.dim_promotion where PromotionName = '4P Beta' " ).first()[0]
invoice_df = spark.read.format("csv").option("header",True).load(rawpath_invoice)

invoice_df = invoice_df.withColumn('PatientClassificationUUID',expr('uuid()'))\
                .withColumn('ListPrice',col('ListPrice').cast("Int"))\
                .withColumn('PromotionUUID',lit(PromotionUUID))\
                .withColumn('Active',lit('1'))\
                .withColumn('CreatedBy',lit('ADB_PatientClassification'))\
                .withColumn('CreatedDate',current_timestamp())\
                .withColumn('UpdatedBy',lit('ADB_PatientClassification'))\
                .withColumn('UpdatedDate',current_timestamp())\
                .select('PatientClassificationUUID','PromotionUUID','PatientClassificationName','DisplayName','PatientClassificationDescription','SKUCode','ListPrice','CreatedDate','CreatedBy','UpdatedDate','UpdatedBy','Active')

invoice_df.display()

In [0]:
targetDF = DeltaTable.forPath(spark, silverpath_patientclassification)  
(targetDF.alias("tgt")
  .merge(invoice_df.alias("src"), "src.PatientClassificationName = tgt.PatientClassificationName and tgt.Active = 1")
  .whenMatchedUpdate(
    set = {
        "tgt.PatientClassificationDescription": "src.PatientClassificationDescription",
        "tgt.PromotionUUID":"src.PromotionUUID", 
        "tgt.ListPrice": "src.ListPrice",
        "tgt.DisplayName": "src.DisplayName",
        "tgt.SKUCode": "src.SKUCode",
        "tgt.Active": "src.Active",
        "tgt.UpdatedDate": "src.UpdatedDate",
        "tgt.UpdatedBy": "src.UpdatedBy"
           }
  )
  .whenNotMatchedInsertAll()
  .execute()
)